# ERA5 fields, climate indices and the daily predictor table

Converts hourly ERA5 to daily regional means, builds Niño 3.4 (lagged one month), DMI and IOBW from OISST, reads the RMM index, and merges everything with the MHW labels into `predictor_table_1982_2025.csv`. The last cells repair the 31 Dec 2025 flux day and verify the rebuilt files.

**Thesis:** Sections 3.1, 3.2 · Appendix C.2, C.3  
**Input:** ERA5 hourly files, OISST, NOAA CPC Niño 3.4, BoM RMM  
**Output:** predictor_table_1982_2025.csv (included in data/processed/)

Runs on Google Colab with the folder layout described in the repository README (`/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/...`). The printed outputs below are from the original run used in the thesis.

#Era5

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import xarray as xr, numpy as np, pandas as pd, os, glob
BASE   = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/raw'
INST = sorted(glob.glob(f'{BASE}/ERA5/temp_u10_v10/ERA5_t2m_u10_v10_*.nc'))   # edit: your t2m/u10/v10 files
ACC  = sorted(glob.glob(f'{BASE}/ERA5/Flux/ERA5_surface_flux_*.nc'))     # edit: your flux files
print(len(INST), len(ACC))
print(xr.open_dataset(INST[0])); print(xr.open_dataset(ACC[0]))

Mounted at /content/drive
44 44
<xarray.Dataset> Size: 1GB
Dimensions:     (time: 12, valid_time: 8760, latitude: 33, longitude: 33)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 1982-01-01 ... 1982-12-31T23...
  * latitude    (latitude) float64 264B 23.0 22.75 22.5 ... 15.5 15.25 15.0
  * longitude   (longitude) float64 264B 85.0 85.25 85.5 ... 92.5 92.75 93.0
    number      int64 8B ...
    expver      (time, valid_time) <U4 2MB ...
Dimensions without coordinates: time
Data variables:
    t2m         (time, valid_time, latitude, longitude) float32 458MB ...
    u10         (time, valid_time, latitude, longitude) float32 458MB ...
    v10         (time, valid_time, latitude, longitude) float32 458MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Foreca

Cell B — Hourly → daily, one file per year

In [2]:
PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
OUT  = f'{PROC}/era5_daily'; os.makedirs(OUT, exist_ok=True)
FLUX = [('ssr','swnet'), ('str','lwnet'), ('slhf','lhf'), ('sshf','shf')]

def op(f, first_only=False):
    ds = xr.open_dataset(f).drop_vars(['expver','number'], errors='ignore')
    if first_only:
        ds = ds.isel(valid_time=[0])
    if 'time' in ds.dims:                                 # collapse the 12 monthly slices
        v0 = list(ds.data_vars)[0]
        chk = ds[v0].isel(latitude=16, longitude=16).notnull().sum('time')
        assert int(chk.min()) == 1 and int(chk.max()) == 1, f'unexpected structure in {f}'
        ds = ds.max('time', skipna=True)
    return ds.rename(valid_time='time').sortby('time')

yr_of = lambda f: pd.Timestamp(xr.open_dataset(f).valid_time.values[0]).year   # reads metadata only
inst = {yr_of(f): f for f in INST}
acc  = {yr_of(f): f for f in ACC}
assert sorted(inst) == sorted(acc) == list(range(1982, 2026))

for yr in range(1982, 2026):
    out = f'{OUT}/era5_daily_{yr}.nc'
    if os.path.exists(out): continue
    a = op(inst[yr]).sel(time=str(yr))
    d = xr.Dataset()
    d['t2m']  = a.t2m.resample(time='1D').mean() - 273.15
    d['u10']  = a.u10.resample(time='1D').mean()
    d['v10']  = a.v10.resample(time='1D').mean()
    d['wspd'] = np.hypot(a.u10, a.v10).resample(time='1D').mean()
    b = op(acc[yr])
    if yr + 1 in acc:
        b = xr.concat([b, op(acc[yr + 1], first_only=True)], dim='time')
    b = b.assign_coords(time=b.time - np.timedelta64(1, 'h')).sel(time=str(yr))
    for src, dst in FLUX:
        d[dst] = b[src].resample(time='1D').mean() / 3600
    d['qnet'] = d.swnet + d.lwnet + d.lhf + d.shf
    d.to_netcdf(out)
    print(yr, 'inst hours:', a.time.size, 'flux hours:', b.time.size,
          'NaN t2m:', int(d.t2m.isnull().sum()), 'NaN qnet:', int(d.qnet.isnull().sum()))

Cell C — One combined daily file

In [3]:
ds = xr.open_mfdataset(sorted(glob.glob(f'{OUT}/era5_daily_*.nc')), combine='by_coords').load()
units = {'t2m':'degC','u10':'m s-1','v10':'m s-1','wspd':'m s-1',
         'swnet':'W m-2','lwnet':'W m-2','lhf':'W m-2','shf':'W m-2','qnet':'W m-2'}
for v, u in units.items(): ds[v].attrs['units'] = u
ds.attrs = {'source':'ERA5 hourly single levels, Copernicus CDS',
            'domain':'15-23N, 85-93E',
            'processing':'daily means 00-23 UTC; fluxes from hourly accumulations /3600; '
                         'positive = downward (into ocean); wspd = daily mean of hourly speed; '
                         '2025-12-31 fluxes from 23 h'}
assert ds.time.size == 16071
enc = {v: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for v in ds.data_vars}
ds.to_netcdf(f'{PROC}/era5_daily_1982_2025_BoB.nc', encoding=enc)
print(ds)

<xarray.Dataset> Size: 630MB
Dimensions:    (time: 16071, latitude: 33, longitude: 33)
Coordinates:
  * time       (time) datetime64[ns] 129kB 1982-01-01 1982-01-02 ... 2025-12-31
  * latitude   (latitude) float64 264B 23.0 22.75 22.5 22.25 ... 15.5 15.25 15.0
  * longitude  (longitude) float64 264B 85.0 85.25 85.5 ... 92.5 92.75 93.0
Data variables:
    t2m        (time, latitude, longitude) float32 70MB 16.1 16.41 ... 24.19
    u10        (time, latitude, longitude) float32 70MB 1.21 1.457 ... 0.5164
    v10        (time, latitude, longitude) float32 70MB -2.097 -1.885 ... -3.653
    wspd       (time, latitude, longitude) float32 70MB 2.49 2.448 ... 3.763
    swnet      (time, latitude, longitude) float32 70MB 163.6 165.8 ... 197.0
    lwnet      (time, latitude, longitude) float32 70MB -106.0 -106.6 ... -88.24
    lhf        (time, latitude, longitude) float32 70MB -35.5 -29.3 ... -138.8
    shf        (time, latitude, longitude) float32 70MB -28.77 -36.01 ... -17.95
    qnet       

Cell D — Regional series on the 805 OISST ocean cells, plus gate checks

In [4]:
OISST_DAY = '/content/drive/MyDrive/Colab Notebooks/oisst_daily.nc'      # edit
sst0  = xr.open_dataset(OISST_DAY).sst
sst0  = sst0.isel(time=0) if 'time' in sst0.dims else sst0
sst0  = sst0.squeeze(drop=True).sel(lat=slice(15, 23), lon=slice(85, 93))
ocean = sst0.notnull()
assert int(ocean.sum()) == 805, int(ocean.sum())

era = xr.open_dataset(f'{PROC}/era5_daily_1982_2025_BoB.nc').sortby('latitude')
g = era.interp(latitude=ocean.lat.values, longitude=ocean.lon.values) \
       .rename(latitude='lat', longitude='lon').where(ocean)
assert int(g.t2m.isel(time=0).notnull().sum()) == 805
w = np.cos(np.deg2rad(ocean.lat))
reg = g.weighted(w).mean(('lat','lon')).to_dataframe()[list(units)]
reg.index = pd.to_datetime(reg.index).normalize()
reg.to_csv(f'{PROC}/era5_regional_daily.csv')

# ---- gate checks ----
print(reg.shape, 'NaN:', int(reg.isna().sum().sum()))
print(reg.describe().round(2))
print(reg[['swnet','lwnet','lhf','shf','qnet']].groupby(reg.index.month).mean().round(1))

(16071, 9) NaN: 0
            t2m       u10       v10      wspd     swnet     lwnet       lhf  \
count  16071.00  16071.00  16071.00  16071.00  16071.00  16071.00  16071.00   
mean      26.96      1.41      1.41      5.62    203.30    -61.02   -124.56   
std        1.78      3.22      3.89      2.19     47.19     24.07     40.90   
min       21.42     -7.86     -8.24      1.44     43.05   -122.05   -480.30   
25%       25.71     -1.38     -2.16      3.94    174.05    -82.55   -148.19   
50%       27.52      1.53      1.86      5.19    201.89    -55.75   -117.01   
75%       28.21      3.85      4.68      6.96    240.62    -38.71    -94.15   
max       30.54     10.59     11.52     16.23    296.93    -24.93    -22.19   

            shf      qnet  
count  16071.00  16071.00  
mean      -9.06      8.66  
std        5.25     77.99  
min      -47.26   -440.78  
25%      -11.91    -48.26  
50%       -8.05     16.74  
75%       -5.50     73.25  
max       13.09    201.11  
      swnet  lwnet

In [5]:
print(reg.qnet.nsmallest(5))

time
2021-05-25   -440.776321
2020-05-19   -390.781150
2021-05-24   -333.601219
1995-11-24   -317.972108
2012-01-13   -313.200635
Name: qnet, dtype: float64


# Climate Indices


Cell E — Niño-3.4 (monthly, lagged one month)

In [6]:
url = 'https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii'
raw = pd.read_csv(url, sep=r'\s+')
per = pd.to_datetime(dict(year=raw.YR, month=raw.MON, day=1)).dt.to_period('M')
n34 = pd.Series(raw.iloc[:, -1].values, index=per)          # last column = Niño-3.4 anomaly
n34.index = n34.index + 1                                    # month m becomes usable in month m+1
days = pd.date_range('1982-01-01', '2025-12-31', freq='D')
nino34 = pd.Series(n34.reindex(days.to_period('M')).values, index=days, name='nino34')
print(nino34.isna().sum(), nino34['1997-12':'1998-02'].resample('MS').first())

0 1997-12-01    2.41
1998-01-01    2.29
1998-02-01    2.38
Freq: MS, Name: nino34, dtype: float64


Cell F — Indian Ocean box means from OISST (long run, restart-safe)

In [7]:
import subprocess
IOD = f'{PROC}/io_boxes'; os.makedirs(IOD, exist_ok=True)
TMP = '/content/oisst_tmp'; os.makedirs(TMP, exist_ok=True)
DL  = 'https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/sst.day.mean.{}.nc'

def boxmean(da, s, n, w, e):
    b = da.sel(lat=slice(s, n), lon=slice(w, e))
    return b.weighted(np.cos(np.deg2rad(b.lat))).mean(('lat','lon')).values

for yr in range(1982, 2026):
    out = f'{IOD}/io_{yr}.csv'
    if os.path.exists(out): continue
    f = f'{TMP}/sst_{yr}.nc'
    r = subprocess.run(['wget', '-q', '-c', '--tries=10', '--waitretry=60', '-O', f, DL.format(yr)])
    if r.returncode != 0:
        raise RuntimeError(f'{yr}: download failed, wget code {r.returncode}')
    with xr.open_dataset(f, engine='netcdf4') as ds:
        sst = ds.sst.sel(lat=slice(-20, 20), lon=slice(40, 110)).load()
    pd.DataFrame({'wtio': boxmean(sst, -10, 10, 50, 70),
                  'seio': boxmean(sst, -10,  0, 90, 110),
                  'iobw': boxmean(sst, -20, 20, 40, 100)},
                 index=pd.to_datetime(sst.time.values)).to_csv(out)
    os.remove(f)
    print(yr, sst.time.size)

Cell G — DMI and IOBW anomalies

In [8]:
io = pd.concat([pd.read_csv(f, index_col=0, parse_dates=True)
                for f in sorted(glob.glob(f'{IOD}/io_*.csv'))])
def anom(s):                                    # 1982–2021 daily climatology, 31-day smoothed
    doy  = s.index.dayofyear - (s.index.is_leap_year & (s.index.month > 2))
    clim = s[s.index.year <= 2021].groupby(doy[s.index.year <= 2021]).mean()
    c = pd.concat([clim.iloc[-15:], clim, clim.iloc[:15]]).rolling(31, center=True).mean().iloc[15:-15]
    c.index = clim.index
    return s - c.reindex(doy).values
dmi  = (anom(io.wtio) - anom(io.seio)).rename('dmi')
ia   = anom(io.iobw)
t    = np.arange(len(ia))
iobw = (ia - np.polyval(np.polyfit(t, ia.values, 1), t)).rename('iobw_dt')   # detrended, like your labels
print(len(dmi), dmi.isna().sum(), dmi['1997-10':'1997-11'].mean().round(2), dmi['2019-10':'2019-11'].mean().round(2))

16071 0 1.52 1.76


In [9]:
ref = pd.read_csv(f'{PROC}/dmi_monthly.csv')
ref.index = pd.to_datetime(dict(year=ref.year, month=ref.month, day=1))
mine = dmi.resample('MS').mean()
both = pd.concat([mine, ref.dmi], axis=1, keys=['oisst', 'hadisst']).dropna()
print(len(both), 'months, correlation:', both.corr().iloc[0, 1].round(3))

528 months, correlation: 0.816


Cell H — MJO (RMM1, RMM2)

In [10]:
import subprocess
RMM = f'{PROC}/rmm.74toRealtime.txt'
if not os.path.exists(RMM) or os.path.getsize(RMM) < 1_000_000:
    subprocess.run(['wget', '-q', '-U', 'Mozilla/5.0', '-O', RMM,
                    'https://www.bom.gov.au/clim_data/IDCKGEM000/rmm.74toRealtime.txt'])
print('file size:', os.path.getsize(RMM) if os.path.exists(RMM) else 'missing')

r = pd.read_csv(RMM, sep=r'\s+', skiprows=2, header=None, usecols=range(7),
                names=['y','m','d','rmm1','rmm2','phase','amp'])
r.index = pd.to_datetime(dict(year=r.y, month=r.m, day=r.d))
rmm = r.loc['1982':'2025', ['rmm1','rmm2']]
print('last date in file:', r.index.max().date())
print(len(rmm), (rmm.abs() > 100).sum().to_dict())   # target: 16071 and {0, 0}

file size: 2735706
last date in file: 2026-10-14
16071 {'rmm1': 0, 'rmm2': 0}


In [11]:
b = both.copy()
m = b.index.month
b = b - b[b.index.year <= 2021].groupby(m[b.index.year <= 2021]).transform('mean').reindex(b.index).values \
      if False else b - b[b.index.year <= 2021].groupby(b[b.index.year <= 2021].index.month).mean().reindex(m).values
print('same baseline:', b.corr().iloc[0, 1].round(3))
t = np.arange(len(b))
bd = b.apply(lambda s: s - np.polyval(np.polyfit(t, s, 1), t))
print('same baseline + detrended:', bd.corr().iloc[0, 1].round(3))
print('Sep-Nov only:', bd[bd.index.month.isin([9, 10, 11])].corr().iloc[0, 1].round(3))

same baseline: 0.818
same baseline + detrended: 0.837
Sep-Nov only: 0.902


#Final merge

In [12]:
# 1. save Cell G results (still in memory)
#pd.concat([dmi, iobw], axis=1).to_csv(f'{PROC}/io_indices_daily.csv')

# 2. load every piece
era  = pd.read_csv(f'{PROC}/era5_regional_daily.csv', index_col=0, parse_dates=True)
io   = pd.read_csv(f'{PROC}/io_indices_daily.csv',   index_col=0, parse_dates=True)

raw  = pd.read_csv('https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii', sep=r'\s+')
n34  = pd.Series(raw.iloc[:, -1].values,
                 index=pd.to_datetime(dict(year=raw.YR, month=raw.MON, day=1)).dt.to_period('M') + 1)
days = pd.date_range('1982-01-01', '2025-12-31', freq='D')
nino = pd.Series(n34.reindex(days.to_period('M')).values, index=days, name='nino34')

r    = pd.read_csv(f'{PROC}/rmm.74toRealtime.txt', sep=r'\s+', skiprows=2, header=None, usecols=range(7),
                   names=['y','m','d','rmm1','rmm2','phase','amp'])
r.index = pd.to_datetime(dict(year=r.y, month=r.m, day=r.d))

hits = glob.glob('/content/drive/MyDrive/MHW/mhw_check/mhw_daily_labels.csv', recursive=True)
print('labels file:', hits)
lab  = pd.read_csv(hits[0], index_col=0, parse_dates=True)
print('label columns:', list(lab.columns))

# 3. merge on date
for d in (era, io, lab): d.index = pd.to_datetime(d.index).normalize()
df = pd.concat([era, io, nino, r[['rmm1','rmm2']], lab], axis=1).loc['1982-01-01':'2025-12-31']
df.to_csv(f'{PROC}/predictor_table_1982_2025.csv')

# 4. gate checks
print(df.shape, '| NaN per column:', df.isna().sum()[df.isna().sum() > 0].to_dict())
print(df.head(3).T)

labels file: ['/content/drive/MyDrive/MHW/mhw_check/mhw_daily_labels.csv']
label columns: ['sst', 'clim_mean', 'thresh90', 'ssta', 'mhw_raw', 'sst_detrended', 'thresh90_detrended', 'mhw_detrended']
(16071, 22) | NaN per column: {}
                    1982-01-01  1982-01-02  1982-01-03
t2m                  22.896380   22.649352   22.632085
u10                  -1.136142   -0.405326   -0.799038
v10                  -4.539334   -3.766417   -3.835092
wspd                  5.172127    4.294859    4.526368
swnet               194.226830  194.479657  188.226525
lwnet              -101.740249 -101.780273  -98.793280
lhf                -181.030612 -158.665626 -158.448294
shf                 -16.967966  -15.937286  -15.540274
qnet               -105.511997  -81.903528  -84.555323
dmi                  -0.303663   -0.503338   -0.488241
iobw_dt              -0.073919   -0.059260   -0.042550
nino34               -0.200000   -0.200000   -0.200000
rmm1                 -1.811420   -1.837960   -1.573200

verification of ERA5 and OISST

In [13]:
df = pd.read_csv(f'{PROC}/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)

print('t2m vs SST correlation:', df.t2m.corr(df.sst).round(3))                     # > 0.9
print('MHW days raw:', int(df.mhw_raw.sum()), '| detrended:', int(df.mhw_detrended.sum()))  # 1652 | 1192
lag = {k: round(df.t2m.corr(df.sst.shift(k)), 3) for k in (-2, -1, 0, 1, 2)}
print('t2m vs SST at lags (days):', lag)                                           # peak at 0 or ±1
print('Nino3.4 Dec97-Feb98:', df.nino34['1997-12-01':'1998-02-28'].resample('MS').first().round(2).tolist())

t2m vs SST correlation: 0.914
MHW days raw: 1652 | detrended: 1192
t2m vs SST at lags (days): {-2: np.float64(0.922), -1: np.float64(0.919), 0: np.float64(0.914), 1: np.float64(0.908), 2: np.float64(0.9)}
Nino3.4 Dec97-Feb98: [2.41, 2.29, 2.38]


#Fixing data problem or limitations

In [14]:
!pip -q install "cdsapi>=0.7.4"
import os
# Put your own CDS API key below (free account: https://cds.climate.copernicus.eu).
# The key used for the thesis was removed before publishing.
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write("url: https://cds.climate.copernicus.eu/api\nkey: YOUR-CDS-API-KEY\n")

import cdsapi
EXTRA = f'{PROC}/era5_flux_20260101T00.nc'
cdsapi.Client().retrieve('reanalysis-era5-single-levels', {
    'product_type': ['reanalysis'],
    'variable': ['surface_latent_heat_flux', 'surface_sensible_heat_flux',
                 'surface_net_solar_radiation', 'surface_net_thermal_radiation'],
    'year': ['2026'], 'month': ['01'], 'day': ['01'], 'time': ['00:00'],
    'data_format': 'netcdf', 'download_format': 'unarchived',
    'area': [23, 85, 15, 93]}, EXTRA)
print(xr.open_dataset(EXTRA))

2026-10-20 05:25:24,859 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

4f0d52b81c45064cdfa556ee6edcdb9e.nc:   0%|          | 0.00/59.7k [00:00<?, ?B/s]

<xarray.Dataset> Size: 18kB
Dimensions:     (valid_time: 1, latitude: 33, longitude: 33)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 2026-01-01
  * latitude    (latitude) float64 264B 23.0 22.75 22.5 ... 15.5 15.25 15.0
  * longitude   (longitude) float64 264B 85.0 85.25 85.5 ... 92.5 92.75 93.0
    number      int64 8B ...
    expver      <U4 16B ...
Data variables:
    slhf        (valid_time, latitude, longitude) float32 4kB ...
    sshf        (valid_time, latitude, longitude) float32 4kB ...
    ssr         (valid_time, latitude, longitude) float32 4kB ...
    str         (valid_time, latitude, longitude) float32 4kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-20T05:25 GRIB to CDM+CF via cfgrib-0.9.1

In [15]:
from google.colab import drive; drive.mount('/content/drive')
import xarray as xr, numpy as np, pandas as pd, os, glob, shutil

PROC  = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
BASE  = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/raw'
INST  = sorted(glob.glob(f'{BASE}/ERA5/temp_u10_v10/ERA5_t2m_u10_v10_*.nc'))
ACC   = sorted(glob.glob(f'{BASE}/ERA5/Flux/ERA5_surface_flux_*.nc'))
EXTRA = f'{PROC}/era5_flux_20260101T00.nc'
FLUX  = [('ssr','swnet'), ('str','lwnet'), ('slhf','lhf'), ('sshf','shf')]
assert os.path.exists(EXTRA), 'run A1 first: the 2026-01-01 00:00 file is missing'

def op(f):
    ds = xr.open_dataset(f).drop_vars(['expver','number'], errors='ignore')
    if 'time' in ds.dims: ds = ds.max('time', skipna=True)
    return ds.rename(valid_time='time').sortby('time')

a = op(INST[-1]).sel(time='2025')
d = xr.Dataset()
d['t2m']  = a.t2m.resample(time='1D').mean() - 273.15
d['u10']  = a.u10.resample(time='1D').mean()
d['v10']  = a.v10.resample(time='1D').mean()
d['wspd'] = np.hypot(a.u10, a.v10).resample(time='1D').mean()
b = xr.concat([op(ACC[-1]), op(EXTRA)[[s for s, _ in FLUX]]], dim='time')
b = b.assign_coords(time=b.time - np.timedelta64(1, 'h')).sel(time='2025')
for src, dst in FLUX:
    d[dst] = b[src].resample(time='1D').mean() / 3600
d['qnet'] = d.swnet + d.lwnet + d.lhf + d.shf
d = d.load()

d.to_netcdf('/content/era5_daily_2025.nc')                                   # local disk
shutil.copy('/content/era5_daily_2025.nc', f'{PROC}/era5_daily/era5_daily_2025.nc')
print('flux hours:', b.time.size, '| NaN:', int(d.to_array().isnull().sum()))  # 8760 | 0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
flux hours: 8760 | NaN: 0


In [16]:
df = pd.read_csv(f'{PROC}/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
anom = lambda s: s - s.groupby(s.index.dayofyear).transform('mean')
tend = anom((df.sst.shift(-1) - df.sst.shift(1)) / 2)        # SST change on day t, °C/day
q    = anom(df.qnet)
r = {k: round(q.corr(tend.shift(-k)), 3) for k in range(-4, 5)}
print(r, '| peak at lag', max(r, key=r.get))

{-4: np.float64(0.007), -3: np.float64(0.073), -2: np.float64(0.168), -1: np.float64(0.266), 0: np.float64(0.312), 1: np.float64(0.277), 2: np.float64(0.187), 3: np.float64(0.093), 4: np.float64(0.027)} | peak at lag 0


In [17]:
cols = ['t2m','wspd','u10','v10','lhf','shf','qnet']
clim = df[cols].groupby(df.index.dayofyear).mean().loc[8:18].mean()
print(df.loc['2012-01-08':'2012-01-18', cols].round(1))
print('mid-January climatology:\n', clim.round(1))

             t2m  wspd  u10  v10    lhf   shf   qnet
2012-01-08  24.3   3.3 -1.7 -1.4  -83.2  -8.5   16.0
2012-01-09  24.2   2.8 -1.3 -1.0  -79.3  -8.1   21.4
2012-01-10  24.0   3.9 -1.8 -2.8 -100.8 -11.1   -7.7
2012-01-11  23.8   5.5 -1.6 -4.8 -159.3 -17.9  -78.7
2012-01-12  23.2   6.7 -0.7 -6.2 -228.4 -27.7 -169.7
2012-01-13  22.1   8.1  0.0 -7.8 -355.3 -46.7 -313.2
2012-01-14  21.4   7.2 -0.3 -6.9 -322.7 -47.3 -282.2
2012-01-15  21.9   6.2 -0.7 -5.7 -257.7 -31.5 -195.8
2012-01-16  22.6   4.4 -0.3 -3.4 -175.7 -18.7  -92.0
2012-01-17  23.1   3.3  0.5 -0.9 -114.8 -10.5  -19.5
2012-01-18  23.3   2.9  0.5 -0.2  -89.9  -8.1   11.5
mid-January climatology:
 t2m      23.6
wspd      4.7
u10      -1.5
v10      -3.3
lhf    -150.7
shf     -13.7
qnet    -61.5
dtype: float64


In [18]:
from google.colab import drive; drive.mount('/content/drive')
import xarray as xr, numpy as np, pandas as pd, os, glob, time

PROC  = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
BASE  = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/raw'
ACC   = sorted(glob.glob(f'{BASE}/ERA5/Flux/ERA5_surface_flux_*.nc'))
EXTRA = f'{PROC}/era5_flux_20260101T00.nc'
res = {}

# 1. files rebuilt in the right order: extra hour -> 2025 daily -> grid -> regional -> table
files = {'extra hour':    EXTRA,
         '2025 daily':    f'{PROC}/era5_daily/era5_daily_2025.nc',
         'combined grid': f'{PROC}/era5_daily_1982_2025_BoB.nc',
         'regional':      f'{PROC}/era5_regional_daily.csv',
         'table':         f'{PROC}/predictor_table_1982_2025.csv'}
mt = [os.path.getmtime(f) for f in files.values()]
for k, m in zip(files, mt): print(f'   {k:14s} {time.strftime("%Y-%m-%d %H:%M", time.gmtime(m))} UTC')
res['1. rebuilt in correct order'] = all(a <= b for a, b in zip(mt, mt[1:]))

# 2. 31 Dec 2025 recomputed from the 24 raw hours equals the gridded file
def op(f):
    ds = xr.open_dataset(f).drop_vars(['expver','number'], errors='ignore')
    if 'time' in ds.dims: ds = ds.max('time', skipna=True)
    return ds.rename(valid_time='time').sortby('time')
V = ['ssr','str','slhf','sshf']
b = xr.concat([op(ACC[-1])[V], op(EXTRA)[V]], dim='time').sel(time=slice('2025-12-31T01:00', '2026-01-01T00:00'))
q_raw = (sum(b[v] for v in V).mean('time') / 3600).sortby('latitude')
q_file = xr.open_dataset(files['combined grid']).sortby('latitude').qnet.sel(time='2025-12-31').squeeze()
diff = float(abs(q_raw - q_file).max())
print(f'   31 Dec 2025: {b.time.size} hours, max difference {diff:.4f} W m-2')
res['2. 31 Dec 2025 uses 24 hours'] = b.time.size == 24 and diff < 0.01

# 3. table carries exactly the rebuilt regional ERA5
df  = pd.read_csv(files['table'],    index_col=0, parse_dates=True)
reg = pd.read_csv(files['regional'], index_col=0, parse_dates=True)
res['3. table = regional ERA5'] = float((df[reg.columns] - reg).abs().max().max()) < 1e-6

# 4-8. earlier gate checks, rerun on the final table
res['4. 16071 x 22, no NaN'] = df.shape == (16071, 22) and int(df.isna().sum().sum()) == 0
res['5. MHW days 1652 / 1192'] = int(df.mhw_raw.sum()) == 1652 and int(df.mhw_detrended.sum()) == 1192
r = df.t2m.corr(df.sst); print(f'   t2m-SST r = {r:.3f}')
res['6. t2m-SST r > 0.9'] = r > 0.9
anom = lambda s: s - s.groupby(s.index.dayofyear).transform('mean')
tend, q = anom((df.sst.shift(-1) - df.sst.shift(1)) / 2), anom(df.qnet)
lag = {k: q.corr(tend.shift(-k)) for k in range(-4, 5)}
res['7. date alignment peak at lag 0'] = max(lag, key=lag.get) == 0
res['8. Nino3.4 lag (Dec97 = 2.41)'] = round(df.nino34['1997-12-15'], 2) == 2.41

print()
for k, v in res.items(): print('PASS' if v else 'FAIL', k)
print('\nALL FIXED' if all(res.values()) else '\nSOMETHING FAILED - send me the output')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   extra hour     2026-10-20 05:25 UTC
   2025 daily     2026-10-20 05:26 UTC
   combined grid  2026-10-20 05:23 UTC
   regional       2026-10-20 05:24 UTC
   table          2026-10-20 05:25 UTC
   31 Dec 2025: 24 hours, max difference 0.0001 W m-2
   t2m-SST r = 0.914

FAIL 1. rebuilt in correct order
PASS 2. 31 Dec 2025 uses 24 hours
PASS 3. table = regional ERA5
PASS 4. 16071 x 22, no NaN
PASS 5. MHW days 1652 / 1192
PASS 6. t2m-SST r > 0.9
PASS 7. date alignment peak at lag 0
PASS 8. Nino3.4 lag (Dec97 = 2.41)

SOMETHING FAILED - send me the output


In [19]:
# ===== CHECK cell: how were the indices and ERA5 fluxes built? (read-only, a few seconds) =====
import os, glob, numpy as np, pandas as pd
PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
df = pd.read_csv(f'{PROC}/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
print('columns:', df.columns.tolist(), '\n')

# 1) daily or monthly? (1 distinct value per month = a monthly value repeated on every day)
mon = df.index.to_period('M')
for v in ['nino34', 'dmi', 'iobw_dt', 'rmm1']:
    if v in df: print(f'{v:8s} median distinct values per month: {df[v].groupby(mon).nunique().median():.0f}')

# 2) if monthly: does day t carry its own month's value (up to a month of future info) or the previous month's?
srcs = {'nino34': [f for f in glob.glob('/content/drive/MyDrive/**/*oni*.csv', recursive=True)],
        'dmi':    [f for f in glob.glob('/content/drive/MyDrive/**/*dmi*.csv', recursive=True)]}
for v, files in srcs.items():
    if v not in df or not files: print(f'{v}: no source file found'); continue
    s = pd.read_csv(files[0]); s.columns = [c.lower() for c in s.columns]
    s.index = pd.PeriodIndex(year=s['year'], month=s['month'], freq='M'); s = s.iloc[:, -1]
    mid = df[v].groupby(mon).median()
    same = np.mean(np.isclose(mid, s.reindex(mid.index), atol=1e-3))
    prev = np.mean(np.isclose(mid, s.reindex(mid.index - 1), atol=1e-3))
    print(f'{v}: source {os.path.basename(files[0])} | matches same month {same:.0%} | matches previous month {prev:.0%}')
    if df[v].groupby(mon).nunique().median() > 1:
        print(f'   {v} varies daily; correlation of its monthly mean with the source: '
              f'{np.corrcoef(mid.dropna(), s.reindex(mid.dropna().index).ffill())[0, 1]:.2f}')

# 3) ERA5 flux units: plausible box means in W m-2 (shortwave ~150-250, latent ~ -100 to -150)
for v in ['swnet', 'lwnet', 'lhf', 'shf', 'qnet', 't2m', 'wspd']:
    if v in df: print(f'{v:6s} mean {df[v].mean():9.2f}   min {df[v].min():9.2f}   max {df[v].max():9.2f}')

columns: ['t2m', 'u10', 'v10', 'wspd', 'swnet', 'lwnet', 'lhf', 'shf', 'qnet', 'dmi', 'iobw_dt', 'nino34', 'rmm1', 'rmm2', 'sst', 'clim_mean', 'thresh90', 'ssta', 'mhw_raw', 'sst_detrended', 'thresh90_detrended', 'mhw_detrended'] 

nino34   median distinct values per month: 1
dmi      median distinct values per month: 31
iobw_dt  median distinct values per month: 31
rmm1     median distinct values per month: 31


/tmp/ipykernel_1613/1959005729.py:18: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  s.index = pd.PeriodIndex(year=s['year'], month=s['month'], freq='M'); s = s.iloc[:, -1]


nino34: source oni_monthly.csv | matches same month 0% | matches previous month 2%
dmi: source dmi_monthly.csv | matches same month 0% | matches previous month 0%
   dmi varies daily; correlation of its monthly mean with the source: 0.81
swnet  mean    203.30   min     43.05   max    296.93
lwnet  mean    -61.02   min   -122.05   max    -24.93
lhf    mean   -124.56   min   -480.30   max    -22.19
shf    mean     -9.06   min    -47.26   max     13.09
qnet   mean      8.66   min   -440.78   max    201.11
t2m    mean     26.96   min     21.42   max     30.54
wspd   mean      5.62   min      1.44   max     16.23


/tmp/ipykernel_1613/1959005729.py:18: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  s.index = pd.PeriodIndex(year=s['year'], month=s['month'], freq='M'); s = s.iloc[:, -1]
